In [11]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from keras import layers
import os
from tensorboard.plugins.hparams import api as hp
from tensorboard import version
from sklearn.model_selection import train_test_split

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("a")


Num GPUs Available:  0
a


In [12]:
df = pd.read_csv('..\\..\\Datasets\\NewDataset\\ProcessedDataset.csv')
print(df.corr()['Severity'].sort_values(ascending=False))

X = df.drop(['Severity'], axis=1)
y = df['Severity']

Severity             1.000000
Air Pollution        0.382265
Alcohol Usage        0.223088
Obesity              0.067149
Passive Smoker       0.052250
Age                  0.035742
Smoking              0.011514
Chest Pain          -0.001281
Genetic Risk        -0.033536
Coughing of Blood   -0.034209
Gender              -0.074792
Lung Disease        -0.101964
Name: Severity, dtype: float64


In [ ]:
nums = []

def run(hparams, num, i): #run_dir,
    #with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)
        accuracy = train_test_model(hparams)
        print(accuracy)
        if (i == 0):
            nums.append(accuracy)
        else: 
            nums[num] += accuracy
        if (i == 4):
            average = nums[num]/5
            nums[num] = average
            print(average)
            #tf.summary.scalar(METRIC_ACCURACY, average, step=1)

In [14]:
def train_test_model(hparams):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(hparams[HP_NUM_UNITS], activation=tf.nn.relu),
        tf.keras.layers.Dropout(hparams[HP_DROPOUT]),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer=hparams[HP_OPTIMIZER],
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=hparams[HP_EPOCHS],
        steps_per_epoch = int(1173/hparams[HP_EPOCHS]),
        verbose = 1,
        callbacks = [
            
        ]
    ) #hp.KerasCallback(direct, hparams), tf.keras.callbacks.TensorBoard(direct)
    _, accuracy = model.evaluate(X_test, y_test)
    #tf.keras.utils.plot_model(model, show_shapes=True, rankdir="LR")
    return accuracy

In [18]:
Permutations = []
for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    HP_NUM_UNITS = hp.HParam('num_units', hp.Discrete(list(range(72, 75, 1)))) #Was 17, 21, 1
    HP_DROPOUT = hp.HParam('dropout', hp.RealInterval(0.0, 0.0)) #hp.RealInterval(0.0, 0.1)
    HP_OPTIMIZER = hp.HParam('optimizer', hp.Discrete(['Adam', 'sgd']))
    HP_EPOCHS = hp.HParam('epochs', hp.Discrete(list(range(52, 55, 1)))) #Was 35, 40, 1
    METRIC_ACCURACY = 'accuracy'
    if (i == 0):
        '''with tf.summary.create_file_writer(direct).as_default():
            hp.hparams_config(
                hparams=[HP_NUM_UNITS, HP_DROPOUT, HP_OPTIMIZER, HP_EPOCHS],
                metrics=[hp.Metric(METRIC_ACCURACY, display_name='Accuracy')],
            )'''
    session_num = 0
    for num_units in HP_NUM_UNITS.domain.values:
        for dropout_rate in (HP_DROPOUT.domain.min_value, HP_DROPOUT.domain.max_value):
            for optimizer in HP_OPTIMIZER.domain.values:
                for epoch in HP_EPOCHS.domain.values:
                    hparams = {
                        HP_NUM_UNITS: num_units,
                        HP_DROPOUT: dropout_rate,
                        HP_OPTIMIZER: optimizer,
                        HP_EPOCHS: epoch
                    }
                    run_name = "run-%d" %  i + "-" + str(session_num)
                    print('--- Starting trial: %s' % run_name)
                    print({h.name: hparams[h] for h in hparams})
                    Permutations.append({h.name: hparams[h] for h in hparams})
                    run(hparams=hparams, num=session_num, i=i) #direct + "/" + 
                    session_num += 1

for trial in nums:
    trial = trial / 5.0

for i in range(len(nums)):
    print(f"Trial {i}: {nums[i]} ")
Best = max(nums)
BestCombination = Permutations[nums.index(Best)]
print(f"Best trial: {Best}")
print(f"Best combination: {BestCombination}")


--- Starting trial: run-0-0
{'num_units': 72, 'dropout': 0.0, 'optimizer': 'Adam', 'epochs': 52}
Epoch 1/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3771 - loss: 1.7485
Epoch 2/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5307 - loss: 1.0467
Epoch 3/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6126 - loss: 0.9006
Epoch 4/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6655 - loss: 0.7918
Epoch 5/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7005 - loss: 0.7163
Epoch 6/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7227 - loss: 0.6751
Epoch 7/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7218 - loss: 0.6564
Epoch 8/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7440 - loss: 0.6136
Epoch 9/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7654 - loss: 0.5797
Epoch 10/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7901 - loss: 0.5246
Epoch 11/52
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy:

In [ ]:
def train_test_tuned_model():
    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(73, activation=tf.nn.relu),
        tf.keras.layers.Dropout(0.0),
        tf.keras.layers.Dense(4, activation=tf.nn.softmax)
    ])
    model.compile(
        optimizer= 'Adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    model.fit(
        X_train,
        y_train,
        epochs=51,
        steps_per_epoch = int(1173/51),
    )
    _, accuracy = model.evaluate(X_test, y_test)
    return accuracy


sum = 0

for i in range(10):
    run_name = "run-%d" % (i+1)
    print('--- Starting trial: %s' % run_name)
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    datasetX = tf.convert_to_tensor(X_train)
    datasetY = tf.convert_to_tensor(y_train)
    data = train_test_tuned_model()
    sum += data
    print(data)
    
average = sum / 10

print(average)


--- Starting trial: run-1
Epoch 1/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4266 - loss: 1.2821
Epoch 2/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5631 - loss: 0.9439
Epoch 3/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6246 - loss: 0.8107
Epoch 4/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6758 - loss: 0.7307
Epoch 5/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6741 - loss: 0.6833
Epoch 6/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7150 - loss: 0.6441
Epoch 7/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7389 - loss: 0.5902
Epoch 8/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7568 - loss: 0.5797
Epoch 9/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7662 - loss: 0.5367
Epoch 10/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7662 - loss: 0.5291
Epoch 11/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7944 - loss: 0.4989
Epoch 12/51
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/st